# 03 — Ablation checkpoints and training-free baselines (no training)
Re-evaluates the nine availability-aware ablation checkpoints (expression-only, expression+spatial, expression+PPI; 3 seeds), the constant/visible-mean baselines (A, B) and the NeighborMean / IDW spatial baselines, then compares with the result tables stored during the revision.
Scope: manuscript Tables 3–5 inputs only. Old-protocol (pre-revision) ablation checkpoints are not part of the release; the legacy columns of Baselines A/B are reproduced from the dataset package.
Ablation checkpoints are read from Drive (`revision_checkpoints_2026-10-18/`). IDW needs the HEST spot coordinates (`hest_expression/st/<slide>.h5ad`).

In [ ]:
# --- Setup: mount Drive, unpack the repository, install dependencies ---
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys, os
BASE = '/content/drive/MyDrive/New-SASP-idea'                 # data/checkpoints live here (not in the repo)
REPO_ZIP = BASE + '/code_release_export_2026-10-23/spatial-sasp-gat.zip'   # or: git clone <repo URL> /content/spatial-sasp-gat
import shutil, hashlib
if os.path.isdir('/content/spatial-sasp-gat'):
    shutil.rmtree('/content/spatial-sasp-gat')          # never reuse an older unpacked copy: always run the zip that is on Drive
subprocess.run(['unzip', '-q', '-o', REPO_ZIP, '-d', '/content'], check=True)
print('repository zip sha256[:16]:', hashlib.sha256(open(REPO_ZIP, 'rb').read()).hexdigest()[:16], '| size', os.path.getsize(REPO_ZIP))
if 'sasp_gat' in sys.modules: print('WARNING: sasp_gat was already imported in this runtime - reset the runtime so the new code is used')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anndata'], check=True)   # reads HEST .h5ad; scikit-learn / networkx are preinstalled on Colab (do NOT reinstall: it can swap pandas under a running kernel)
# data_tables exported in ENV-2/ENV-3 (copied only if missing in the repo)
import shutil
for f in ['sasp54_genes_canonical_order.csv', 'ppi_string_v12_edges.csv', 'slide_partition.csv']:
    dst = '/content/spatial-sasp-gat/data_tables/' + f
    if not os.path.exists(dst):
        shutil.copy(BASE + '/code_release_export_2026-10-23/' + f, dst)
sys.path.insert(0, '/content/spatial-sasp-gat/src')
import torch, torch_geometric
print('torch', torch.__version__, '| torch_geometric', torch_geometric.__version__,
      '| cuda', torch.cuda.is_available())

In [ ]:
import hashlib, json, pandas as pd, numpy as np, torch
from pathlib import Path
from sasp_gat.config import *
from sasp_gat.data import load_package, split_samples, sha256_file
from sasp_gat.ppi import ppi_edge_index_from_networkx
from sasp_gat.availability import build_availability, effective_mask, score_mask
from sasp_gat.evaluation import load_model, eval_both
from sasp_gat.training import evaluate_multimask_av

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPO = Path('/content/spatial-sasp-gat'); T = REPO / 'data_tables'
PKL = Path(BASE) / 'MODEL_READY_DATASET_v1_2026-09-26.pkl'
pkg, pyg_dataset, gene_order = load_package(PKL)
train_s, val_s, test_s, hd_s = split_samples(pyg_dataset)
AVAIL, LOGGED = build_availability(pyg_dataset.keys(), gene_order, T / 'availability_unmeasured.csv')
ppi_ei = ppi_edge_index_from_networkx(pkg['ppi_graph_networkx'], gene_order, device)
CKDIR = REPO / 'checkpoints' if (REPO / 'checkpoints' / 'full_avail_seed42.pt').exists() else Path(BASE) / 'revision_checkpoints_2026-10-18'
print('checkpoint dir:', CKDIR)
import torch_geometric, scipy; print('versions: torch', torch.__version__, '| pyg', torch_geometric.__version__, '| numpy', np.__version__, '| pandas', pd.__version__, '| scipy', scipy.__version__)
import warnings; warnings.filterwarnings('ignore')
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'anndata'])   # only needed to read HEST .h5ad coordinates
from sasp_gat.training import vkey
from sasp_gat.baselines import (train_gene_means, legacy_gene_means, eval_baseline, build_neighbour_edges,
                                eval_neighbour_methods)
ABL_DIR = Path(BASE) / 'revision_checkpoints_2026-10-18'
TAB_DIR = Path(BASE) / 'revision_tables_2026-10-18'
BL_DIR  = Path(BASE) / 'revision_spatial_baselines_2026-10-18'
VARIANTS = ['expression-only', 'expression+spatial', 'expression+PPI']
ms = lambda s: f'{s.mean():.4f} ± {s.std(ddof=1):.4f}'

In [ ]:
# 1) Ablation checkpoints: hash-free integrity (metadata) + validation reproduction, then locked test / HD
rows = []; ABL = {}
for vn in VARIANTS:
    for seed in SEEDS:
        p = ABL_DIR / f'{vkey(vn)}_avail_seed{seed}.pt'
        m, ck = load_model(p, device, expect_seed=seed, expect_avail=True)
        assert ck['variant_name'] == vn, (p.name, ck['variant_name'])
        val = evaluate_multimask_av(m, val_s, AVAIL, ppi_ei, device, role='val')
        assert abs(val - float(ck['best_val_loss'])) < 1e-4, (p.name, val, ck['best_val_loss'])
        t = eval_both(m, test_s, AVAIL, ppi_ei, device, 'test', input_av=True)
        h = eval_both(m, hd_s,   AVAIL, ppi_ei, device, 'hd',   input_av=True)
        assert t['n_old'] - t['n_av'] == 1980 and h['n_old'] == h['n_av']
        rows.append(dict(variant=vn, seed=seed, val=val, test_av=t['mse_av'], hd_av=h['mse_av']))
ABL = pd.DataFrame(rows)
print(ABL.round(6).to_string(index=False))
for vn in VARIANTS:
    a = ABL[ABL.variant == vn]
    print(f'{vn:19s} val {ms(a.val)} | TEST {ms(a.test_av)} | HD {ms(a.hd_av)}')
print('validation losses reproduced for all 9 checkpoints (|diff| < 1e-4); 1,980 unmeasured test targets excluded in each')

In [ ]:
# 2) Compare with the stored seed-level table (new-protocol rows only; columns are printed if the layout differs)
t3 = pd.read_csv(TAB_DIR / 'table3_seed_level_test_hd_all_models.csv')
print('stored table3 columns:', list(t3.columns))
if {'model', 'variant', 'seed', 'test_av', 'hd_av'} <= set(t3.columns):
    st = t3[(t3.model == 'yeni') & t3.variant.isin(VARIANTS)][['variant', 'seed', 'test_av', 'hd_av']]
    mg = ABL.merge(st, on=['variant', 'seed'], suffixes=('', '_stored'))
    assert len(mg) == 9, len(mg)
    mg['d_test'] = (mg.test_av - mg.test_av_stored).abs(); mg['d_hd'] = (mg.hd_av - mg.hd_av_stored).abs()
    print(mg[['variant', 'seed', 'test_av', 'test_av_stored', 'hd_av', 'hd_av_stored']].round(6).to_string(index=False))
    print('max |diff| vs stored: test %.2e | HD %.2e ->' % (mg.d_test.max(), mg.d_hd.max()), 'MATCH' if max(mg.d_test.max(), mg.d_hd.max()) < 1e-5 else 'DIFFERENT')
else:
    print('stored layout differs from the expected one; paste this output so the comparison can be adapted.'); print(t3.head())

In [ ]:
# 3) Baselines A and B under the three scoring definitions
train_x = train_s
mu_old = legacy_gene_means(train_s).to(device)
mu_new = train_gene_means(train_s, AVAIL).to(device)
cohorts = {'test': test_s, 'hd': hd_s}
rows = []
for kind in ['A', 'B']:
    for coh, smp in cohorts.items():
        old = eval_baseline(kind, smp, AVAIL, coh, mu_old, False, device)
        new = eval_baseline(kind, smp, AVAIL, coh, mu_new, True, device)
        rows.append(dict(baseline=kind, cohort=coh, eski_skor=old['mse_old'], score_only=old['mse_av'], tam_av=new['mse_av'],
                         n_removed=old['n_old'] - old['n_av'], n_total=old['n_old']))
BA = pd.DataFrame(rows); print(BA.round(6).to_string(index=False))
# legacy (original-protocol) values stored in the original results CSV
oldb = pd.read_csv(Path(BASE) / 'model_checkpoints_2026-09-27' / 'final_baseline_comparison_2026-09-27.csv').set_index('metric')['value']
d = max(abs(BA[(BA.baseline == k) & (BA.cohort == c)].eski_skor.iloc[0] - oldb[f'baseline_{k}_{c}']) for k in 'AB' for c in ('test', 'hd'))
print('original baseline CSV reproduced, max |diff| = %.2e' % d); assert d < 1e-5
t5 = pd.read_csv(TAB_DIR / 'table5_baselines_three_scorings.csv'); print('stored table5 columns:', list(t5.columns))
if {'baseline', 'cohort', 'eski_skor', 'score_only', 'tam_av'} <= set(t5.columns):
    mg = BA.merge(t5, on=['baseline', 'cohort'], suffixes=('', '_stored'))
    dd = max((mg[c] - mg[c + '_stored']).abs().max() for c in ['eski_skor', 'score_only', 'tam_av'])
    print('max |diff| vs stored table5 = %.2e ->' % dd, 'MATCH' if dd < 1e-5 else 'DIFFERENT')

In [ ]:
# 4) NeighborMean and IDW (1-hop). Needs HEST coordinates; graph = dataset edge_index; fallback = mu_new
H5_DIR = Path(BASE) / 'hest_expression' / 'st'
EDG = build_neighbour_edges(cohorts, H5_DIR, device)
# leakage / sensitivity check on one slide: hidden inputs (random-masked or unmeasured) must not change predictions
from sasp_gat.baselines import neighbour_predict
from sasp_gat.masking import make_mask
sid = next(iter(test_s)); d0 = test_s[sid]; x = d0.x.to(device); av = AVAIL[sid].to(device)
mk = make_mask(x.shape[0], mask_ratio=MASK_RATIO, deterministic_seed_str=f'{sid}_test_0', device=device)
hid = mk | (~av).unsqueeze(0)
for k, (s_, t_, w_) in EDG[sid].items():
    p0, _ = neighbour_predict(x, mk, av, s_, t_, w_, mu_new.double())
    xh = x.clone(); xh[hid] = 1e6; p1, _ = neighbour_predict(xh, mk, av, s_, t_, w_, mu_new.double())
    xv = x.clone(); xv[~hid] = 1e6; p2, _ = neighbour_predict(xv, mk, av, s_, t_, w_, mu_new.double())
    print(f'  {k:13s}: hidden-input effect {(p0-p1).abs().max().item():.1e} | visible-input effect {(p0-p2).abs().max().item():.1e}')
    assert (p0 - p1).abs().max().item() < 1e-9 and (p0 - p2).abs().max().item() > 1.0
RES = eval_neighbour_methods(cohorts, EDG, AVAIL, mu_new, device)
assert RES[(RES.cohort == 'test') & (RES.method == 'NeighborMean')].n.sum() == 1791286 - 1980
TAB = (RES.groupby(['cohort', 'method']).apply(lambda z: pd.Series(dict(pooled=z.ss.sum() / z.n.sum(), fallback_pct=100 * z.n_fb.sum() / z.n.sum()))).reset_index())
print(TAB.round(5).to_string(index=False))
ps = pd.read_csv(BL_DIR / 'spatial_baselines_pooled_summary.csv'); print('stored columns:', list(ps.columns))
if {'cohort', 'method', 'pooled'} <= set(ps.columns):
    mg = TAB.merge(ps, on=['cohort', 'method'], suffixes=('', '_stored'))
    dd = (mg.pooled - mg.pooled_stored).abs().max()
    print('max |diff| vs stored = %.2e ->' % dd, 'MATCH' if dd < 1e-6 else 'DIFFERENT')
print('(three masks are re-draws on the same slides, not independent replicates; no SD is reported for these baselines)')